# Redbus Journey Analysis

In [88]:
!pip install pandas numpy matplotlib seaborn scikit-learn tensorflow
!pip install xgboost lightgbm catboost optuna -q

In [89]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [90]:
#Load path where my dataset is
path = "/content/drive/MyDrive/RedBusAnalysis"

In [91]:
#Importing pandas and numpy
import numpy as np
import pandas as pd

In [92]:
#Loading data set
train_data = pd.read_csv(path + "/train.csv")
transaction_data = pd.read_csv(path + "/transactions.csv")
test_data = pd.read_csv(path + "/test_8gqdJqH.csv")

# Feature Extraction

In [93]:
#Filtering for prediction 15 days before journey
transaction_15 = transaction_data[transaction_data["dbd"] == 15]

In [94]:
#Creating unique route key to match later with test dataset
transaction_15["route_key"] = transaction_15["doj"] + "_" + transaction_15["srcid"].astype(str) + "_" + transaction_15["destid"].astype(str)

/tmp/ipython-input-94-1703587420.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  transaction_15["route_key"] = transaction_15["doj"] + "_" + transaction_15["srcid"].astype(str) + "_" + transaction_15["destid"].astype(str)


In [95]:
transaction_15 = transaction_15.dropna()

In [96]:
#Selecting Relevant feature
features = transaction_15[["route_key", "cumsum_seatcount", "cumsum_searchcount", "srcid_region", "destid_region", "srcid_tier", "destid_tier"]]

In [97]:
#Merge with train labels
train_data["route_key"] = train_data["doj"] + "_" + train_data["srcid"].astype(str) + "_" + train_data["destid"].astype(str)
#Drop if existing feature in train data to avoid collision during merge
cols_to_drop = ["cumsum_seatcount", "cumsum_searchcount",
                "srcid_region", "destid_region", "srcid_tier", "destid_tier"]

train_data = train_data.drop(columns=[col for col in cols_to_drop if col in train_data.columns])

train_data = train_data.merge(features, on="route_key", how="left")
train_data.dropna(inplace=True)

In [98]:
#Mergin with test set
duplicate_cols = [
    "cumsum_seatcount", "cumsum_searchcount",
    "srcid_region", "destid_region",
    "srcid_tier", "destid_tier"
]

# Drop them from test_data if they exist
test_data = test_data.drop(columns=[col for col in duplicate_cols if col in test_data.columns])
test_data = test_data.merge(features, on="route_key", how="left")
test_data['cumsum_seatcount'] = test_data['cumsum_seatcount'].fillna(0)
test_data['cumsum_searchcount'] = test_data['cumsum_searchcount'].fillna(0)
for col in ["srcid_region", "destid_region", "srcid_tier", "destid_tier"]:
    test_data[col] = test_data[col].fillna("Unknown")

In [99]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [100]:
# Encoding categorical features safely
categorical_features = ["srcid_region", "destid_region", "srcid_tier", "destid_tier"]
for col in categorical_features:
    le = LabelEncoder()

    # Combine train + test categories for fitting
    combined_values = pd.concat([train_data[col], test_data[col]], axis=0).astype(str)

    # Fit encoder on all possible values
    le.fit(combined_values)

    # Transform separately
    train_data[col] = le.transform(train_data[col].astype(str))
    test_data[col] = le.transform(test_data[col].astype(str))

In [101]:
#Select final features
features = ["cumsum_seatcount", "cumsum_searchcount"]
target = "final_seatcount"

In [102]:
X = train_data[features]
y = train_data[target]
X_test = test_data[features]

In [104]:
#Normalizing features
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

# Model Training

In [105]:
#Split data for Optuna
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [106]:
#Optuna Hyperparameter Tuning for XGBoost
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
import optuna

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }
    model = XGBRegressor(**params, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    return rmse

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

best_params = study.best_params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)
xgb_pred = best_model.predict(X_test)


[I 2025-06-21 08:31:51,050] A new study created in memory with name: no-name-945a36f4-c84e-48fa-bd09-f3ff10233435
[I 2025-06-21 08:31:54,331] Trial 0 finished with value: 847.4847460751029 and parameters: {'n_estimators': 933, 'max_depth': 10, 'learning_rate': 0.05113512143119261, 'subsample': 0.9640130767039091, 'colsample_bytree': 0.7297686555564799}. Best is trial 0 with value: 847.4847460751029.
[I 2025-06-21 08:31:54,786] Trial 1 finished with value: 849.9969388101621 and parameters: {'n_estimators': 304, 'max_depth': 3, 'learning_rate': 0.22316115113730955, 'subsample': 0.6545802079387396, 'colsample_bytree': 0.871306161454204}. Best is trial 0 with value: 847.4847460751029.
[I 2025-06-21 08:31:58,959] Trial 2 finished with value: 847.297792585028 and parameters: {'n_estimators': 635, 'max_depth': 7, 'learning_rate': 0.06741741896449337, 'subsample': 0.9222832934798177, 'colsample_bytree': 0.9720310462353969}. Best is trial 2 with value: 847.297792585028.
[I 2025-06-21 08:31:59,8

# Sequential input preparation

In [107]:
import numpy as np
import pandas as pd

# Filter only needed columns
ts_data = transaction_data[["doj", "srcid", "destid", "dbd", "cumsum_seatcount", "cumsum_searchcount"]]

# Create route_key
ts_data["route_key"] = ts_data["doj"] + "_" + ts_data["srcid"].astype(str) + "_" + ts_data["destid"].astype(str)

# Pivot to get a fixed-length time series per route_key
ts_pivot = ts_data.pivot_table(index="route_key", columns="dbd", values=["cumsum_seatcount", "cumsum_searchcount"])

# Drop rows with missing values (incomplete sequences)
ts_pivot = ts_pivot.dropna()

# Convert MultiIndex columns to flat names
ts_pivot.columns = [f"{var}_dbd{day}" for var, day in ts_pivot.columns]

# Merge with train_data to get final seat counts
lstm_train = train_data[["route_key", "final_seatcount"]].merge(ts_pivot, on="route_key", how="inner")

X_lstm = lstm_train.drop(columns=["route_key", "final_seatcount"]).values
y_lstm = lstm_train["final_seatcount"].values

# Reshape for LSTM input (samples, time steps, features)
X_lstm = X_lstm.reshape((X_lstm.shape[0], 2, X_lstm.shape[1] // 2))


# LSTM Training

In [108]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

# Train-test split
X_train_lstm, X_val_lstm, y_train_lstm, y_val_lstm = train_test_split(X_lstm, y_lstm, test_size=0.2, random_state=42)

# Define LSTM model
model = Sequential([
    LSTM(64, input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2]), return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(loss='mse', optimizer=Adam(0.001))
model.fit(X_train_lstm, y_train_lstm, epochs=50, batch_size=32, validation_data=(X_val_lstm, y_val_lstm))


Epoch 1/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 3938087.5000 - val_loss: 1052786.7500
Epoch 2/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 875089.6875 - val_loss: 510011.6875
Epoch 3/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 550940.8750 - val_loss: 350316.6875
Epoch 4/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 520726.4062 - val_loss: 393436.1250
Epoch 5/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 534255.1875 - val_loss: 369671.1562
Epoch 6/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 22s 7ms/step - loss: 528255.5000 - val_loss: 592659.2500
Epoch 7/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 19s 6ms/step - loss: 663940.6250 - val_loss: 484620.9375
Epoch 8/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 664600.4375 - val_loss: 521076.0625
Epoch 9/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 780027.8125 - val_loss: 1411825.6250
Epoch 10/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 1320210.8750 - val_loss: 900760.6875
Epoch

In [113]:
X_test_lstm = test_data[features].values
X_test_lstm = X_test_lstm.reshape((X_test_lstm.shape[0], 2, X_test_lstm.shape[1] // 2))

In [114]:
lstm_pred = model.predict(X_test_lstm).flatten()

185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [120]:
#Model 2: LGBMRegressor
from lightgbm import LGBMRegressor
lgb = LGBMRegressor(n_estimators=100, random_state=42)
lgb.fit(X_train, y_train)
lgb_pred = lgb.predict(X_test)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001276 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 53760, number of used features: 2
[LightGBM] [Info] Start training from score 2003.632533


# Building Ensemble Model

In [122]:
final_pred = 0.4 * xgb_pred + 0.3 * lgb_pred + 0.3 * lstm_pred

In [125]:
submission = test_data[["route_key"]].copy()
submission["final_seatcount"] = final_pred.round().astype(int)
submission.to_csv("submission_file.csv", index=False)

In [126]:
#Download submission file
from google.colab import files
files.download("submission_file.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [127]:
submission

,route_key,final_seatcount
0,2025-02-11_46_45,3062
1,2025-01-20_17_23,2135
2,2025-01-08_02_14,1419
3,2025-01-08_08_47,1419
4,2025-01-08_09_46,1419
...,...,...
5895,2025-01-23_46_48,3326
5896,2025-02-21_46_09,1419
5897,2025-01-17_32_19,1952
5898,2025-01-24_45_03,1419
